# BE175 CGM Data Labeling Notebook

This notebook converts cleaned CGM data into a labeled machine learning dataset for the BE175 final project.

**Input file format:**

`patient_id | timestamp_numeric | glucose`

**Output file format:**

Each output row represents one prediction time point. The model inputs are the previous 60 minutes of CGM readings, and the label tells whether sustained hypoglycemia occurs in the next 40 minutes.

**Label definition used here:**

`label = 1` if there are at least 3 consecutive future CGM readings below 70 mg/dL within the next 40 minutes.

Because CGM readings are about every 5 minutes, 3 consecutive readings represent about 15 minutes of sustained hypoglycemia.

**Important:** the future columns are only used to create/check the label. They should not be used as model input features.


## STEP 1: Import packages and set file names

For testing, use the shorter CSV file first. For the full dataset, change `INPUT_FILE` and set `N_ROWS = None`.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# File settings
INPUT_FILE = "clean_cgm_data_full.csv"
OUTPUT_FILE = "labeled_cgm_full.csv"

# Use 1000 for testing, or None to read the full file
N_ROWS = None

# Timestamp gap settings
# Most readings should be around 300 seconds apart, or 5 minutes.
# Values outside this range start a new continuous segment.
GAP_MIN_SECONDS = 240
GAP_MAX_SECONDS = 360

# Safety tolerance for checking the full past/future span
TIME_TOLERANCE_SECONDS = 90


## STEP 2: Load the cleaned CGM CSV

This reads only the three columns needed for labeling. The data types help reduce memory use when using the full dataset.


In [ ]:
dtype_map = {
    "patient_id": "int32",
    "timestamp_numeric": "int64",
    "glucose": "float32",
}

required_cols = ["patient_id", "timestamp_numeric", "glucose"]

df = pd.read_csv(
    INPUT_FILE,
    usecols=required_cols,
    dtype=dtype_map,
    nrows=N_ROWS,
)

# Drop rows with missing key values, just in case
df = df.dropna(subset=required_cols)

# Sort by patient and time
df = df.sort_values(["patient_id", "timestamp_numeric"]).reset_index(drop=True)

print("Input shape:", df.shape)
print("Columns:", list(df.columns))
display(df.head())


## STEP 3: Check timestamp spacing and create continuous segments

This prevents a 60-minute input window or 40-minute labeling window from crossing a missing-data gap.

The data usually has timestamps about 300 seconds apart. If a time difference is too small or too large, the code starts a new `segment_id`.


In [ ]:
# Calculate time differences within each patient
df["time_diff_seconds"] = df.groupby("patient_id")["timestamp_numeric"].diff()

print("Most common time differences:")
print(df["time_diff_seconds"].value_counts(dropna=False).head(10))

# Mark abnormal gaps. The first row of each patient has NaN and should start a new segment.
abnormal_gap_mask = (
    df["time_diff_seconds"].notna()
    & (
        (df["time_diff_seconds"] < GAP_MIN_SECONDS)
        | (df["time_diff_seconds"] > GAP_MAX_SECONDS)
    )
)

print("Number of abnormal gaps:", int(abnormal_gap_mask.sum()))

gap_preview_cols = ["patient_id", "timestamp_numeric", "glucose", "time_diff_seconds"]
display(df.loc[abnormal_gap_mask, gap_preview_cols].head(20))

# Start a new segment at the first row of each patient or after an abnormal gap
new_segment = df["time_diff_seconds"].isna() | abnormal_gap_mask

df["segment_id"] = new_segment.groupby(df["patient_id"]).cumsum().astype("int32")

# Remove temporary checking column
df = df.drop(columns=["time_diff_seconds"])

print("Number of segments:", df[["patient_id", "segment_id"]].drop_duplicates().shape[0])
print("Rows per segment, first 10 segments:")
print(df.groupby(["patient_id", "segment_id"]).size().head(10))

display(df.head())


## STEP 4: Define the labeling function

This function creates:

1. Past 60-minute glucose window: `glucose_t_minus_55` through `glucose_t_0`
2. Simple features: rolling mean and slope
3. Future 40-minute checking columns
4. Final binary label

Rows are removed if they do not have enough continuous past and future data.


In [ ]:
def make_labeled_cgm_table(df, time_tolerance_seconds=90):
    df = df.copy()

    needed_cols = {"patient_id", "timestamp_numeric", "glucose", "segment_id"}
    missing_cols = needed_cols - set(df.columns)
    if missing_cols:
        raise ValueError(f"Missing required columns: {missing_cols}")

    # Sort by patient, segment, and time
    df = df.sort_values(["patient_id", "segment_id", "timestamp_numeric"]).reset_index(drop=True)

    # Group by patient and segment so windows never cross gaps
    g = df.groupby(["patient_id", "segment_id"], group_keys=False)

    # Start output table
    out = df[["patient_id", "timestamp_numeric"]].copy()

    # Past 60-minute glucose window: 12 readings from t-55 to t
    past_specs = [
        ("glucose_t_minus_55", 11),
        ("glucose_t_minus_50", 10),
        ("glucose_t_minus_45", 9),
        ("glucose_t_minus_40", 8),
        ("glucose_t_minus_35", 7),
        ("glucose_t_minus_30", 6),
        ("glucose_t_minus_25", 5),
        ("glucose_t_minus_20", 4),
        ("glucose_t_minus_15", 3),
        ("glucose_t_minus_10", 2),
        ("glucose_t_minus_5", 1),
        ("glucose_t_0", 0),
    ]

    for col_name, lag_steps in past_specs:
        if lag_steps == 0:
            out[col_name] = df["glucose"]
        else:
            out[col_name] = g["glucose"].shift(lag_steps)

    past_cols = [col_name for col_name, lag_steps in past_specs]

    # Simple feature: average glucose over the past 60-minute window
    out["glucose_rolling_mean_60min"] = out[past_cols].mean(axis=1)

    # Simple feature: slope from the oldest to current glucose value
    # Units are mg/dL per minute.
    out["glucose_slope_60min"] = (
        out["glucose_t_0"] - out["glucose_t_minus_55"]
    ) / 55

    # Future 40-minute glucose values for labeling only
    future_cols = []
    for minutes_ahead in range(5, 45, 5):
        col_name = f"future_glucose_t_plus_{minutes_ahead}"
        future_cols.append(col_name)
        out[col_name] = g["glucose"].shift(-(minutes_ahead // 5))

    # Checking column: minimum glucose in the next 40 minutes
    out["future_min_glucose_40min"] = out[future_cols].min(axis=1)

    # Label: 1 if there are 3 consecutive future readings below 70 mg/dL
    sustained_hypo = pd.Series(False, index=out.index)
    for i in range(len(future_cols) - 2):
        three_in_a_row_below_70 = (
            (out[future_cols[i]] < 70)
            & (out[future_cols[i + 1]] < 70)
            & (out[future_cols[i + 2]] < 70)
        )
        sustained_hypo = sustained_hypo | three_in_a_row_below_70

    out["future_sustained_hypo_15min_within_40min"] = sustained_hypo
    out["label"] = out["future_sustained_hypo_15min_within_40min"].astype("int8")

    # Final safety check that the past and future windows have the expected time span
    past_span_seconds = df["timestamp_numeric"] - g["timestamp_numeric"].shift(11)
    future_span_seconds = g["timestamp_numeric"].shift(-8) - df["timestamp_numeric"]

    valid_past_window = (past_span_seconds - 55 * 60).abs() <= time_tolerance_seconds
    valid_future_window = (future_span_seconds - 40 * 60).abs() <= time_tolerance_seconds

    final_cols = [
        "patient_id",
        "timestamp_numeric",
        "glucose_t_minus_55",
        "glucose_t_minus_50",
        "glucose_t_minus_45",
        "glucose_t_minus_40",
        "glucose_t_minus_35",
        "glucose_t_minus_30",
        "glucose_t_minus_25",
        "glucose_t_minus_20",
        "glucose_t_minus_15",
        "glucose_t_minus_10",
        "glucose_t_minus_5",
        "glucose_t_0",
        "glucose_rolling_mean_60min",
        "glucose_slope_60min",
        "future_min_glucose_40min",
        "label",
    ]

    out = out.loc[valid_past_window & valid_future_window, final_cols]
    out = out.dropna().reset_index(drop=True)

    return out


## STEP 5: Run the labeling function

This creates the final 18-column labeled dataframe.


In [ ]:
labeled_df = make_labeled_cgm_table(
    df,
    time_tolerance_seconds=TIME_TOLERANCE_SECONDS,
)

print("Original rows:", len(df))
print("Labeled rows:", len(labeled_df))
print("Labeled shape:", labeled_df.shape)

display(labeled_df.head())


## STEP 6: Check the label distribution

This tells you how many rows are labeled as future hypoglycemia risk.

A strong imbalance is expected because hypoglycemia events are less common than non-events.


In [ ]:
print("Label counts:")
print(labeled_df["label"].value_counts())

print()
print("Label proportions:")
print(labeled_df["label"].value_counts(normalize=True))

## STEP 7: Save the labeled CSV

This creates the final output file to send to the modeling group member.


In [ ]:
labeled_df.to_csv(OUTPUT_FILE, index=False)
print(f"Saved labeled file to: {OUTPUT_FILE}")


## STEP 8: Define model feature columns and target

Use these columns for modeling. Do not use the future columns as model inputs because they contain information from after the prediction time.


In [ ]:
FEATURE_COLS = [
    "glucose_t_minus_55",
    "glucose_t_minus_50",
    "glucose_t_minus_45",
    "glucose_t_minus_40",
    "glucose_t_minus_35",
    "glucose_t_minus_30",
    "glucose_t_minus_25",
    "glucose_t_minus_20",
    "glucose_t_minus_15",
    "glucose_t_minus_10",
    "glucose_t_minus_5",
    "glucose_t_0",
    "glucose_rolling_mean_60min",
    "glucose_slope_60min",
]

TARGET_COL = "label"

X = labeled_df[FEATURE_COLS]
y = labeled_df[TARGET_COL]

print("X shape:", X.shape)
print("y shape:", y.shape)


In [ ]:
# STEP 9: Save a model-ready file without future checking columns

model_ready_df = labeled_df[["patient_id", "timestamp_numeric"] + FEATURE_COLS + [TARGET_COL]]

model_ready_output_file = OUTPUT_FILE.replace(".csv", "_model_ready.csv")

model_ready_df.to_csv(model_ready_output_file, index=False)

print("Saved model-ready file to:", model_ready_output_file)
print("Model-ready shape:", model_ready_df.shape)

In [ ]:
# STEP 10: Save a smaller patient subset for quick testing

RANDOM_SEED = 175
N_PATIENTS_SUBSET = 30

patient_ids = model_ready_df["patient_id"].drop_duplicates().to_numpy()

rng = np.random.default_rng(RANDOM_SEED)
chosen_patients = rng.choice(
    patient_ids,
    size=min(N_PATIENTS_SUBSET, len(patient_ids)),
    replace=False
)

small_model_ready_df = model_ready_df[model_ready_df["patient_id"].isin(chosen_patients)]

small_output_file = "model_ready_cgm_30patient_subset.csv"

small_model_ready_df.to_csv(small_output_file, index=False)

print("Saved smaller model-ready file to:", small_output_file)
print("Chosen patients:", chosen_patients)
print("Small model-ready shape:", small_model_ready_df.shape)
print(small_model_ready_df["label"].value_counts())
print(small_model_ready_df["label"].value_counts(normalize=True))